# Data Preparation and Final Merge

In [ ]:
import pandas as pd 
customers_cleaned = pd.read_csv("cleaned_supplyguard_data/customers_cleaned.csv")
shipments_cleaned = pd.read_csv("cleaned_supplyguard_data/shipments_cleaned.csv")
deliveries_cleaned = pd.read_csv("cleaned_supplyguard_data/deliveries_cleaned.csv")
drivers_cleaned = pd.read_csv("cleaned_supplyguard_data/drivers_cleaned.csv")
exchange_rates_cleaned = pd.read_csv("cleaned_supplyguard_data/exchange_rates_cleaned.csv")
vehicles_cleaned = pd.read_csv("cleaned_supplyguard_data/vehicles_cleaned.csv")
warehouses_cleaned = pd.read_csv("cleaned_supplyguard_data/warehouses_cleaned.csv")
delivery_labels_cleaned = pd.read_csv("cleaned_supplyguard_data/delivery_labels_cleaned.csv")

In [ ]:
#Merge all datasets 
master = customers_cleaned.merge(shipments_cleaned, on="customer_id", how="left")
master = master.merge(deliveries_cleaned, on="shipment_id", how="left")
master = master.merge(drivers_cleaned, on="driver_id", how= "left")
master = master.merge(vehicles_cleaned, on="vehicle_id", how="left")
master = master.merge(delivery_labels_cleaned, on="delivery_id", how="left")
master = master.merge(warehouses_cleaned, on="warehouse_id", how="left")


# 6.3 Operational Risk Scoring 


In [ ]:
#Create the late delivery flag
# Identify delieveries that take longer than the 75th percentile 
# of all delivery durations and flag them as late deliveries 
master["late_delivery_flag"] =(master["delivery_duration_hours"] > master["delivery_duration_hours"].quantile(0.75).astype(int))
master["very_long_delivery_flag"] = (master["delivery_distance_km"] > master["delivery_distance_km"].quantile(0.90).astype(int))

In [ ]:
# Count the number of late deliveries asscoiated with each customer 
# Customers in the top 10% of late delivery counts as flagged 
customer_late_count = (master.groupby("customer_id")["late_delivery_flag"].transform("sum"))
customer_delay_threshold = customer_late_count.quantile(0.90)
master["repeated_customer_delay_flag"] = (customer_late_count >= customer_delay_threshold).astype(int)

In [ ]:
# Identify long delivery duration 
# Identify unusually long duration. 
# The top 10% of delivery duration are classified as long duration deliveries 
master["long_delivery_duration_flag"] =(master["delivery_duration_hours"] > master["delivery_duration_hours"].quantile(0.90).astype(int))

In [ ]:
#Identify high risk routes
# Caculate the historical late-delivery rate for each route 
# Routes in the top 10% of the late delivery rates are classified as high routes 
master["route"] = (master["city_x"].astype(str) + "-" + master["city_y"].astype(str))

route_late_rate = (master.groupby("route")["late_delivery_flag"].transform("mean"))
route_threshold = route_late_rate.quantile(0.90)
master ["high_risk_route_flag"] = (route_late_rate > route_threshold).astype(int)

In [ ]:
#Caculate the operational risk score 
#Each risk factor contributes a predefined number of points: 
    #Late Delivery = 25 points 
    # Very long delivery distance = 15 points
    # Repeated customer delays = 20 points 
    # Long delivery duration = 20 points 
    # High risk route = 20 points 

#Maximum possible score = 100 points 

master["risk_score"] =(
    master["late_delivery_flag"] * 25 +
    master["very_long_delivery_flag"] * 15 +
    master["repeated_customer_delay_flag"] * 20 + 
    master["long_delivery_duration_flag"] * 20 +
    master["high_risk_route_flag"] * 20
)

In [ ]:
master["risk_score"].head()

In [ ]:
#Classify each delivery according to its operational risk score. 
#0-30 - Low Risk 
# 31-60 Medium Riks 
# 61 -100 High Risk 

#Classify risk based on score 
def classify_risk(score): 
    if score <= 30:
        return "Low Risk"
    elif score <=60:
        return "Medium Risk"
    else: 
        return "High Risk"

master["risk_category"] = master["risk_score"].apply(classify_risk)

In [ ]:
master[["risk_score", "risk_category"]].head(10)

UPDATE at 16:09 on 23/28/2026

In [ ]:
#Check the number of deliveries in each operational risk category 
master["risk_category"].value_counts()

In [ ]:
#Caculate the percentage of deliveries in each operational risk category 
master["risk_category"].value_counts(normalize=True).mul(100).round(2)

In [ ]:
#Sumarise the operational risk categories by count and percentages 

risk_summary = (
    master["risk_category"].value_counts().rename_axis("Risk Category").reset_index(name= "Number of Deliveries"))

risk_summary ["Percentage"] = (
    risk_summary["Number of Deliveries"] / risk_summary["Number of Deliveries"].sum() *100).round(2)

risk_summary

In [ ]:
# Display the results in a bar chart.
import matplotlib.pyplot as plt 


risk_counts = master["risk_category"].value_counts()
# plt.figure(figsize=(8, 5))

ax = risk_counts.plot(kind="bar")

plt.title("Distribution of Operational Risk Category Distribution")
plt.xlabel("Risk Category")
plt.ylabel("Number of Deliveries")
plt.xticks(rotation=0)

for i, value in enumerate(risk_counts):
    ax.text(i, value, f"{value:,}", ha = "center", va= "bottom")
plt.tight_layout()
plt.show()

UPDATE 17:10 PM at 23/08/2026